# Visit with Us – Wellness Tourism Purchase Prediction
## End-to-End Machine Learning and MLOps Project

**Objective:** Build a reproducible classification workflow that estimates a customer's propensity to purchase a tourism package, then prepare a GitHub Actions CI/CD pipeline and a Streamlit application.

**Dataset:** `tourism.csv` (4,128 rows; 21 columns in the supplied file).

### Business and modeling caveat
The supplied target `ProdTaken` records historical purchase behaviour; it does not explicitly identify purchases of the newly introduced Wellness Tourism Package. The model therefore estimates historical package-purchase propensity as a proxy, not a verified Wellness-package label.

The business statement asks for prediction *before contacting* a customer. The primary model therefore excludes interaction-stage fields (`DurationOfPitch`, `NumberOfFollowups`, `ProductPitched`, `PitchSatisfactionScore`, and `TypeofContact`). This avoids relying on information that may not exist at targeting time. `TypeofContact` is also excluded because it describes a contact channel rather than an intrinsic customer attribute.

**Execution:** Run every cell from top to bottom in Google Colab. The notebook creates a local project folder containing validation, preparation, training, Streamlit, requirements, README, and GitHub Actions files. The actual training metrics are calculated when you run the notebook; no metrics are pre-filled or fabricated.

## 1. Environment setup

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
# Install the core libraries. Restart the runtime only if Colab requests it.
%pip -q install "numpy>=1.24,<2.3" "pandas==2.2.3" scikit-learn matplotlib seaborn joblib mlflow streamlit

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

**Colab compatibility note:** NumPy is constrained below 2.3 and pandas is pinned to 2.2.3 to match the Colab runtime and Numba requirements. If Colab requests a runtime restart after installation, restart once and continue from the next cell.

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
import os, sys, json, warnings, subprocess, textwrap, shutil, joblib, platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.base import clone

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
PROJECT_DIR = Path("/content/Visit-With-Us-MLOps") if "google.colab" in sys.modules else Path.cwd() / "Visit-With-Us-MLOps"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", platform.python_version())
print("Project directory:", PROJECT_DIR)

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

## 2. Load and register the dataset

**Purpose:** Read the CSV from the repository-style `data/` folder. In Colab, upload `tourism.csv` if it is not already present. The validation checks required columns, target values, row/column counts, missing values and duplicates.

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)
DATA_PATH = DATA_DIR / "tourism.csv"

if not DATA_PATH.exists():
    try:
        from google.colab import files
        print("Upload the supplied tourism.csv file.")
        uploaded = files.upload()
        uploaded_name = next((name for name in uploaded if name.lower().endswith(".csv")), None)
        if uploaded_name is None:
            raise FileNotFoundError("No CSV file was uploaded.")
        shutil.copy(uploaded_name, DATA_PATH)
    except ImportError:
        # For local Jupyter, put tourism.csv in the project data/ directory.
        raise FileNotFoundError(f"Place tourism.csv at {DATA_PATH} and rerun this cell.")

df = pd.read_csv(DATA_PATH)
EXPECTED_COLUMNS = [
    "CustomerID", "ProdTaken", "Age", "TypeofContact", "CityTier",
    "DurationOfPitch", "Occupation", "Gender", "NumberOfPersonVisiting",
    "NumberOfFollowups", "ProductPitched", "PreferredPropertyStar",
    "MaritalStatus", "NumberOfTrips", "Passport", "PitchSatisfactionScore",
    "OwnCar", "NumberOfChildrenVisiting", "Designation", "MonthlyIncome"
]
missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing expected columns: {missing_columns}")
if not set(df["ProdTaken"].dropna().unique()).issubset({0, 1}):
    raise ValueError("ProdTaken must contain only binary values 0 and 1.")

print(f"Dataset registered: {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("Duplicate rows:", int(df.duplicated().sum()))
print("Missing values:", int(df.isna().sum().sum()))
print("Target counts:")
display(df["ProdTaken"].value_counts().rename(index={0:"Not purchased",1:"Purchased"}).to_frame("count"))
display(df.head())

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

## 3. Data understanding and exploratory analysis

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
print("Shape:", df.shape)
display(pd.DataFrame({
    "Data type": df.dtypes.astype(str),
    "Missing count": df.isna().sum(),
    "Unique values": df.nunique()
}))
display(df.describe(include="all").T)

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
# Remove the exported dataframe index if present; retain CustomerID only for traceability, not modeling.
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])
df = df.drop_duplicates().copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x="ProdTaken", ax=axes[0])
axes[0].set_title("Target class distribution")
axes[0].set_xticklabels(["Not purchased (0)", "Purchased (1)"])
sns.histplot(data=df, x="Age", hue="ProdTaken", bins=25, multiple="stack", ax=axes[1])
axes[1].set_title("Age distribution by purchase outcome")
plt.tight_layout()
plt.show()

target_pct = df["ProdTaken"].value_counts(normalize=True).sort_index() * 100
print("Target percentages:")
display(target_pct.rename(index={0:"Not purchased",1:"Purchased"}).round(2).to_frame("percentage"))

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
numeric_all = df.select_dtypes(include=np.number).columns.tolist()
numeric_all = [c for c in numeric_all if c not in ["CustomerID", "ProdTaken"]]
if numeric_all:
    df[numeric_all].hist(figsize=(15, 10), bins=25, edgecolor="black")
    plt.suptitle("Numerical feature distributions")
    plt.tight_layout()
    plt.show()

categorical_all = df.select_dtypes(include=["object"]).columns.tolist()
for col in categorical_all:
    plt.figure(figsize=(8, 3))
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order)
    plt.title(f"Distribution: {col}")
    plt.tight_layout()
    plt.show()

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

### EDA observations to complete after execution
- The target distribution is imbalanced; the positive class is the smaller class. Accuracy alone can hide poor detection of buyers.
- Review numerical histograms and category frequencies above for skewness, rare categories and implausible values.
- The supplied file has an exported index column (`Unnamed: 0`), which is removed. `CustomerID` is retained only as an identifier and excluded from model features.
- The dataset has no exact duplicate rows and no missing values in the supplied snapshot; the pipeline still includes imputers so it remains robust to future incoming data.

## 4. Define the prediction point and feature set

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
TARGET = "ProdTaken"
ID_COLUMNS = ["CustomerID"]
# Fields observed during/after a sales pitch are not available at pre-contact targeting time.
POST_CONTACT_OR_CHANNEL = [
    "TypeofContact", "DurationOfPitch", "NumberOfFollowups",
    "ProductPitched", "PitchSatisfactionScore"
]
FEATURE_COLUMNS = [c for c in df.columns if c not in [TARGET] + ID_COLUMNS + POST_CONTACT_OR_CHANNEL]
X = df[FEATURE_COLUMNS].copy()
y = df[TARGET].astype(int).copy()

print("Features used for pre-contact prediction:")
print(FEATURE_COLUMNS)
print("\nExcluded identifier:", ID_COLUMNS)
print("Excluded post-contact/channel fields:", POST_CONTACT_OR_CHANNEL)
print("Positive class rate:", round(y.mean() * 100, 2), "%")

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

**Modeling decision:** This is a pre-contact propensity model. The excluded fields may be useful for a separate post-pitch conversion model, but including them here would not match the stated operational decision point. This distinction should be retained in the final business recommendations.

## 5. Train/test split and preprocessing

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

**Why this preprocessing?** Median imputation is robust to skewed numeric variables; most-frequent imputation is suitable for categorical variables. Standardization supports scale-sensitive estimators. One-hot encoding prevents the model from interpreting category labels as numeric rankings. The preprocessing is fitted inside each cross-validation fold to avoid leakage.

## 6. Model experimentation and hyperparameter tuning

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
# Small, reproducible search spaces keep Colab execution practical.
# All estimators are wrapped with the same preprocessing pipeline.
model_searches = {
    "Decision Tree": (
        DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
        {"model__max_depth": [3, 6, None], "model__min_samples_leaf": [2, 5]}
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1),
        {"model__n_estimators": [150, 250], "model__max_depth": [8, None],
         "model__min_samples_leaf": [2, 5]}
    ),
    "Bagging": (
        BaggingClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
                          random_state=RANDOM_STATE, n_jobs=-1),
        {"model__n_estimators": [50, 100], "model__max_samples": [0.7, 1.0]}
    ),
    "AdaBoost": (
        AdaBoostClassifier(random_state=RANDOM_STATE),
        {"model__n_estimators": [50, 100], "model__learning_rate": [0.05, 0.5, 1.0]}
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {"model__n_estimators": [75, 125], "model__learning_rate": [0.05, 0.1],
         "model__max_depth": [2, 3]}
    )
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
search_results = []
best_estimators = {}
for model_name, (estimator, param_grid) in model_searches.items():
    print(f"\nTuning {model_name} ...")
    pipe = Pipeline(steps=[("preprocessor", clone(preprocessor)), ("model", estimator)])
    search = GridSearchCV(
        pipe, param_grid=param_grid, scoring="average_precision",
        cv=cv, n_jobs=-1, refit=True, verbose=0, error_score="raise"
    )
    search.fit(X_train, y_train)
    best_estimators[model_name] = search.best_estimator_
    search_results.append({
        "Model": model_name,
        "Best CV PR-AUC": search.best_score_,
        "Best parameters": search.best_params_
    })
    print("Best CV PR-AUC:", round(search.best_score_, 4))
    print("Best parameters:", search.best_params_)

tuning_results = pd.DataFrame(search_results).sort_values("Best CV PR-AUC", ascending=False)
display(tuning_results)

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

**Experiment tracking:** The table records the best cross-validation PR-AUC and all selected hyperparameters for each model. PR-AUC is prioritized because the positive purchase class is relatively rare. The untouched test set is used only after model selection.

## 7. Evaluate candidate models on the holdout test set

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
def get_scores(estimator, X_eval):
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(X_eval)[:, 1]
    if hasattr(estimator, "decision_function"):
        raw = estimator.decision_function(X_eval)
        return (raw - raw.min()) / (raw.max() - raw.min() + 1e-9)
    return estimator.predict(X_eval).astype(float)

evaluation_rows = []
test_scores = {}
for name, estimator in best_estimators.items():
    scores = get_scores(estimator, X_test)
    preds = (scores >= 0.5).astype(int)
    test_scores[name] = scores
    evaluation_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1": f1_score(y_test, preds, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, scores),
        "PR-AUC": average_precision_score(y_test, scores)
    })

evaluation = pd.DataFrame(evaluation_rows).sort_values("PR-AUC", ascending=False)
display(evaluation.style.format({c:"{:.4f}" for c in evaluation.columns if c != "Model"}))

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
# Select by holdout PR-AUC as a ranking criterion; inspect recall/precision trade-off before deployment.
best_model_name = evaluation.iloc[0]["Model"]
best_model = best_estimators[best_model_name]
best_scores = test_scores[best_model_name]
best_predictions = (best_scores >= 0.5).astype(int)

print("Selected model by test PR-AUC:", best_model_name)
print("\nClassification report at default threshold 0.50:")
print(classification_report(y_test, best_predictions, zero_division=0))
print("Confusion matrix:")
cm = confusion_matrix(y_test, best_predictions)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted 0", "Predicted 1"],
            yticklabels=["Actual 0", "Actual 1"])
plt.title(f"Confusion Matrix – {best_model_name}")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
for name, estimator in best_estimators.items():
    RocCurveDisplay.from_predictions(y_test, test_scores[name], name=name, ax=ax)
plt.title("ROC curves – candidate models")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
for name, scores in test_scores.items():
    PrecisionRecallDisplay.from_predictions(y_test, scores, name=name, ax=ax)
plt.title("Precision–Recall curves – candidate models")
plt.tight_layout()
plt.show()

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

### Evaluation interpretation
- Recall measures the proportion of actual buyers identified. Low recall means missed sales opportunities.
- Precision measures how many targeted customers actually purchased. Low precision increases unnecessary outreach.
- F1 balances precision and recall; PR-AUC summarizes precision/recall performance across thresholds.
- The final operating threshold should be selected with marketing capacity and the relative cost of false positives versus false negatives in mind. A default threshold of 0.50 is shown for transparent baseline reporting, not asserted as the business-optimal threshold.

## 8. Select an operating threshold and save the model

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
# Demonstrate a threshold trade-off. The business should approve the final threshold.
threshold_rows = []
for threshold in [0.20, 0.30, 0.40, 0.50, 0.60]:
    pred = (best_scores >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "Customers flagged": int(pred.sum())
    })
threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.round(4))

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)
model_path = MODEL_DIR / "best_model.joblib"
model_bundle = {
    "pipeline": best_model,
    "threshold": 0.50,
    "model_name": best_model_name,
    "feature_columns": FEATURE_COLUMNS,
    "target": TARGET,
    "excluded_columns": POST_CONTACT_OR_CHANNEL,
    "random_state": RANDOM_STATE
}
joblib.dump(model_bundle, model_path)
print("Saved model bundle:", model_path)
print("Model size (KB):", round(model_path.stat().st_size / 1024, 2))

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

## 9. Experiment tracking with MLflow

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
# Local MLflow tracking. This creates a reproducible experiment log in the project directory.
import mlflow
from mlflow.models import infer_signature

mlflow.set_tracking_uri((PROJECT_DIR / "mlruns").resolve().as_uri())
mlflow.set_experiment("Visit-With-Us-Wellness-Propensity")

chosen_row = evaluation.loc[evaluation["Model"] == best_model_name].iloc[0].to_dict()
chosen_params = best_model.named_steps["model"].get_params()
# Keep only scalar parameters for MLflow logging.
with mlflow.start_run(run_name=f"best-{best_model_name.replace(' ', '-')}" ):
    mlflow.log_param("selected_model", best_model_name)
    mlflow.log_param("prediction_stage", "pre-contact")
    mlflow.log_param("selection_metric", "holdout_PR_AUC")
    for key, value in chosen_params.items():
        if isinstance(value, (str, int, float, bool)) or value is None:
            mlflow.log_param(f"model_{key}", str(value))
    for metric in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]:
        mlflow.log_metric(metric.lower().replace("-", "_"), float(chosen_row[metric]))
    mlflow.log_artifact(str(model_path))
print("MLflow run logged. Tracking directory:", PROJECT_DIR / "mlruns")

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

## 10. Prepare reusable MLOps scripts for GitHub Actions

The following cell writes standalone scripts into `src/`. GitHub Actions will call these scripts in separate jobs. The workflow passes train/test CSV files as an artifact between jobs and commits the trained model back to the repository using the repository's `GITHUB_TOKEN`.

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
SRC_DIR = PROJECT_DIR / "src"
APP_DIR = PROJECT_DIR / "app"
WF_DIR = PROJECT_DIR / ".github" / "workflows"
for folder in [SRC_DIR, APP_DIR, WF_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

(SRC_DIR / "ml_pipeline.py").write_text('\nimport argparse, json, os\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.tree import DecisionTreeClassifier\nfrom sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier\nfrom sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,\n                             roc_auc_score, average_precision_score)\n\nROOT = Path(__file__).resolve().parents[1]\nDATA = ROOT / "data" / "tourism.csv"\nART = ROOT / "artifacts"\nMODELS = ROOT / "models"\nTARGET = "ProdTaken"\nID_COLS = ["CustomerID"]\nEXCLUDED = ["TypeofContact", "DurationOfPitch", "NumberOfFollowups",\n            "ProductPitched", "PitchSatisfactionScore"]\nEXPECTED = ["CustomerID","ProdTaken","Age","TypeofContact","CityTier","DurationOfPitch",\n            "Occupation","Gender","NumberOfPersonVisiting","NumberOfFollowups",\n            "ProductPitched","PreferredPropertyStar","MaritalStatus","NumberOfTrips",\n            "Passport","PitchSatisfactionScore","OwnCar","NumberOfChildrenVisiting",\n            "Designation","MonthlyIncome"]\n\ndef validate():\n    if not DATA.exists():\n        raise FileNotFoundError(f"Expected dataset at {DATA}")\n    df = pd.read_csv(DATA)\n    missing = sorted(set(EXPECTED) - set(df.columns))\n    if missing:\n        raise ValueError(f"Missing expected columns: {missing}")\n    if not set(df[TARGET].dropna().unique()).issubset({0,1}):\n        raise ValueError("ProdTaken must be binary (0/1)")\n    print(json.dumps({"rows":len(df),"columns":len(df.columns),\n                      "missing_values":int(df.isna().sum().sum()),\n                      "duplicate_rows":int(df.duplicated().sum()),\n                      "target_counts":df[TARGET].value_counts().to_dict()}, indent=2))\n\ndef prepare():\n    df = pd.read_csv(DATA)\n    df = df.drop(columns=["Unnamed: 0"], errors="ignore").drop_duplicates()\n    X = df.drop(columns=[TARGET] + ID_COLS + EXCLUDED, errors="ignore")\n    y = df[TARGET].astype(int)\n    X_train, X_test, y_train, y_test = train_test_split(\n        X, y, test_size=0.20, random_state=42, stratify=y)\n    ART.mkdir(exist_ok=True)\n    train = X_train.copy(); train[TARGET] = y_train.values\n    test = X_test.copy(); test[TARGET] = y_test.values\n    train.to_csv(ART / "train.csv", index=False)\n    test.to_csv(ART / "test.csv", index=False)\n    (ART / "feature_columns.json").write_text(json.dumps(list(X.columns), indent=2))\n    print(f"Saved train={train.shape}, test={test.shape} to {ART}")\n\ndef train():\n    train_df = pd.read_csv(ART / "train.csv")\n    test_df = pd.read_csv(ART / "test.csv")\n    X_train, y_train = train_df.drop(columns=[TARGET]), train_df[TARGET].astype(int)\n    X_test, y_test = test_df.drop(columns=[TARGET]), test_df[TARGET].astype(int)\n    num = X_train.select_dtypes(include=np.number).columns.tolist()\n    cat = X_train.select_dtypes(exclude=np.number).columns.tolist()\n    prep = ColumnTransformer([\n        ("num", Pipeline([("imputer",SimpleImputer(strategy="median")),\n                          ("scaler",StandardScaler())]), num),\n        ("cat", Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),\n                          ("onehot",OneHotEncoder(handle_unknown="ignore"))]), cat)\n    ])\n    specs = {\n      "Decision Tree": (DecisionTreeClassifier(random_state=42,class_weight="balanced"),\n                        {"model__max_depth":[3,6,None],"model__min_samples_leaf":[2,5]}),\n      "Random Forest": (RandomForestClassifier(random_state=42,class_weight="balanced",n_jobs=-1),\n                        {"model__n_estimators":[150,250],"model__max_depth":[8,None],\n                         "model__min_samples_leaf":[2,5]}),\n      "Bagging": (BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42),\n                                    random_state=42,n_jobs=-1),\n                  {"model__n_estimators":[50,100],"model__max_samples":[0.7,1.0]}),\n      "AdaBoost": (AdaBoostClassifier(random_state=42),\n                   {"model__n_estimators":[50,100],"model__learning_rate":[0.05,0.5,1.0]}),\n      "Gradient Boosting": (GradientBoostingClassifier(random_state=42),\n                            {"model__n_estimators":[75,125],"model__learning_rate":[0.05,0.1],\n                             "model__max_depth":[2,3]})\n    }\n    cv = StratifiedKFold(n_splits=3,shuffle=True,random_state=42)\n    rows=[]; candidates={}\n    for name,(est,grid) in specs.items():\n        pipe=Pipeline([("preprocessor",prep),("model",est)])\n        search=GridSearchCV(pipe,grid,scoring="average_precision",cv=cv,n_jobs=-1,refit=True)\n        search.fit(X_train,y_train)\n        candidates[name]=search.best_estimator_\n        rows.append({"model":name,"cv_pr_auc":float(search.best_score_),\n                     "best_params":{k:str(v) for k,v in search.best_params_.items()}})\n    results=[]\n    for name,model in candidates.items():\n        scores=model.predict_proba(X_test)[:,1]\n        pred=(scores>=0.5).astype(int)\n        results.append({"model":name,"accuracy":float(accuracy_score(y_test,pred)),\n          "precision":float(precision_score(y_test,pred,zero_division=0)),\n          "recall":float(recall_score(y_test,pred,zero_division=0)),\n          "f1":float(f1_score(y_test,pred,zero_division=0)),\n          "roc_auc":float(roc_auc_score(y_test,scores)),\n          "pr_auc":float(average_precision_score(y_test,scores))})\n    best=max(results,key=lambda r:r["pr_auc"])\n    model=candidates[best["model"]]\n    MODELS.mkdir(exist_ok=True)\n    features=list(X_train.columns)\n    bundle={"pipeline":model,"threshold":0.50,"model_name":best["model"],\n            "feature_columns":features,"target":TARGET,\n            "excluded_columns":EXCLUDED,"random_state":42}\n    joblib.dump(bundle,MODELS/"best_model.joblib")\n    (ROOT/"artifacts").mkdir(exist_ok=True)\n    (ROOT/"artifacts"/"training_results.json").write_text(json.dumps(\n       {"cross_validation":rows,"test_evaluation":results,"selected_model":best},indent=2))\n    print(json.dumps({"selected_model":best,"all_results":results},indent=2))\n\nif __name__ == "__main__":\n    parser=argparse.ArgumentParser()\n    parser.add_argument("command",choices=["validate","prepare","train"])\n    args=parser.parse_args()\n    {"validate":validate,"prepare":prepare,"train":train}[args.command]()\n')
(APP_DIR / "streamlit_app.py").write_text('\nimport streamlit as st\nimport pandas as pd\nimport joblib\nfrom pathlib import Path\n\nst.set_page_config(page_title="Visit with Us | Wellness Package", page_icon="✈️", layout="centered")\nst.title("✈️ Visit with Us")\nst.subheader("Wellness Tourism Package – Customer Propensity")\nst.write("Estimate historical package-purchase propensity from customer attributes. This is a propensity estimate, not a guarantee of purchase or a Wellness-specific trained outcome.")\n\nMODEL_PATH = Path(__file__).resolve().parents[1] / "models" / "best_model.joblib"\nif not MODEL_PATH.exists():\n    st.error("Model file not found. Ensure models/best_model.joblib is committed to the repository.")\n    st.stop()\nbundle = joblib.load(MODEL_PATH)\nmodel = bundle["pipeline"]\nthreshold = float(bundle.get("threshold", 0.5))\nfeatures = bundle["feature_columns"]\n\nwith st.form("customer_form"):\n    age = st.number_input("Age", min_value=18, max_value=100, value=35)\n    city_tier = st.selectbox("City tier", [1, 2, 3], index=1)\n    occupation = st.selectbox("Occupation", ["Salaried", "Small Business", "Large Business", "Free Lancer"])\n    gender = st.selectbox("Gender", ["Male", "Female"])\n    persons = st.number_input("Number of people visiting", min_value=1, max_value=20, value=2)\n    preferred_star = st.selectbox("Preferred property star", [3.0, 4.0, 5.0], index=1)\n    marital = st.selectbox("Marital status", ["Single", "Married", "Divorced"])\n    trips = st.number_input("Number of trips per year", min_value=0.0, max_value=50.0, value=2.0, step=1.0)\n    passport = st.selectbox("Has passport?", [0, 1], format_func=lambda x: "Yes" if x else "No")\n    own_car = st.selectbox("Owns a car?", [0, 1], format_func=lambda x: "Yes" if x else "No")\n    children = st.number_input("Children below age 5 visiting", min_value=0, max_value=10, value=0)\n    designation = st.selectbox("Designation", ["Executive", "Manager", "Senior Manager", "AVP", "VP"])\n    income = st.number_input("Monthly income", min_value=0.0, max_value=500000.0, value=25000.0, step=1000.0)\n    submitted = st.form_submit_button("Predict propensity")\n\nif submitted:\n    row = {\n      "Age":float(age), "CityTier":int(city_tier), "Occupation":occupation,\n      "Gender":gender, "NumberOfPersonVisiting":int(persons),\n      "PreferredPropertyStar":float(preferred_star), "MaritalStatus":marital,\n      "NumberOfTrips":float(trips), "Passport":int(passport), "OwnCar":int(own_car),\n      "NumberOfChildrenVisiting":int(children), "Designation":designation,\n      "MonthlyIncome":float(income)\n    }\n    input_df = pd.DataFrame([{col: row.get(col, None) for col in features}])\n    probability = float(model.predict_proba(input_df)[:,1][0])\n    prediction = int(probability >= threshold)\n    st.metric("Estimated purchase propensity", f"{probability:.1%}")\n    if prediction:\n        st.success("Higher propensity segment – consider a relevant, permission-based wellness offer.")\n    else:\n        st.info("Lower propensity at the current threshold. This does not mean the customer will never purchase.")\n    st.caption(f"Model: {bundle.get(\'model_name\',\'saved model\')} | Decision threshold: {threshold:.2f}")\n')
(PROJECT_DIR / "requirements.txt").write_text('pandas\nnumpy\nscikit-learn\njoblib\nstreamlit\nmatplotlib\nseaborn\nmlflow\n')
(WF_DIR / "pipeline.yml").write_text('\nname: Visit with Us MLOps Pipeline\n\non:\n  push:\n    branches: [ main ]\n  workflow_dispatch:\n\npermissions:\n  contents: write\n\njobs:\n  validate:\n    runs-on: ubuntu-latest\n    steps:\n      - name: Checkout repository\n        uses: actions/checkout@v4\n      - name: Set up Python\n        uses: actions/setup-python@v5\n        with:\n          python-version: "3.11"\n      - name: Install dependencies\n        run: pip install -r requirements.txt\n      - name: Validate registered dataset\n        run: python src/ml_pipeline.py validate\n\n  prepare:\n    needs: validate\n    runs-on: ubuntu-latest\n    steps:\n      - uses: actions/checkout@v4\n      - uses: actions/setup-python@v5\n        with:\n          python-version: "3.11"\n      - run: pip install -r requirements.txt\n      - name: Prepare train and test datasets\n        run: python src/ml_pipeline.py prepare\n      - name: Upload prepared data artifact\n        uses: actions/upload-artifact@v4\n        with:\n          name: prepared-data\n          path: artifacts/\n\n  train:\n    needs: prepare\n    runs-on: ubuntu-latest\n    steps:\n      - uses: actions/checkout@v4\n      - uses: actions/setup-python@v5\n        with:\n          python-version: "3.11"\n      - run: pip install -r requirements.txt\n      - name: Download prepared data artifact\n        uses: actions/download-artifact@v4\n        with:\n          name: prepared-data\n          path: artifacts/\n      - name: Train, tune and evaluate models\n        run: python src/ml_pipeline.py train\n      - name: Upload model and evaluation artifact\n        uses: actions/upload-artifact@v4\n        with:\n          name: trained-model\n          path: |\n            models/best_model.joblib\n            artifacts/training_results.json\n      - name: Commit updated model and metrics to main\n        run: |\n          git config user.name "github-actions[bot]"\n          git config user.email "41898282+github-actions[bot]@users.noreply.github.com"\n          git add models/best_model.joblib artifacts/training_results.json\n          git diff --cached --quiet || git commit -m "Update trained model [skip ci]"\n          git push origin HEAD:main\n')
(PROJECT_DIR / "README.md").write_text('# Visit with Us – Wellness Tourism MLOps\n\n## Overview\nPre-contact customer purchase-propensity model using the supplied `ProdTaken` label as a proxy for historical package purchase behaviour. It is not a Wellness-package-specific label.\n\n## Run locally\n```bash\npip install -r requirements.txt\npython src/ml_pipeline.py validate\npython src/ml_pipeline.py prepare\npython src/ml_pipeline.py train\nstreamlit run app/streamlit_app.py\n```\n\n## GitHub Actions\nPush the repository to GitHub with the default branch named `main`. Enable Actions and ensure the repository Actions setting permits read/write access to repository contents. The workflow validates the CSV, creates train/test files, passes them as an artifact, trains and evaluates models, and commits the updated model and evaluation JSON.\n\n## Streamlit Community Cloud\nCreate a public GitHub repository, deploy `app/streamlit_app.py` from the repository root, and ensure `models/best_model.joblib` is committed. Add the public Streamlit URL here after deployment.\n\n## Data governance and prediction timing\nThe primary model excludes contact-channel and pitch interaction fields because the intended prediction is made before contacting the customer. `ProdTaken` is a historical purchase label and is only a proxy for Wellness package propensity.\n')
(PROJECT_DIR / ".gitignore").write_text("".join([
    "__pycache__/\n", ".venv/\n", "mlruns/\n", ".DS_Store\n",
    "artifacts/train.csv\n", "artifacts/test.csv\n"
]))
print("Generated repository files:")
for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

## 11. Run the generated pipeline scripts locally

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
# Validate and prepare through the same scripts that GitHub Actions executes.
subprocess.run([sys.executable, str(SRC_DIR / "ml_pipeline.py"), "validate"], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable, str(SRC_DIR / "ml_pipeline.py"), "prepare"], cwd=PROJECT_DIR, check=True)
print("Local validation and preparation completed.")

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

The CI workflow uses the same scripts. Its training job receives the prepared-data artifact from the preparation job. The workflow commits the model and evaluation report to `main`. The `[skip ci]` commit suffix prevents the model commit from triggering an endless workflow loop.

## 12. Streamlit app and deployment instructions

1. Create a **public** GitHub repository named `Visit-With-Us-MLOps`.
2. Upload/push the complete contents of the generated `Visit-With-Us-MLOps` folder, including the original CSV under `data/tourism.csv`.
3. In GitHub, open **Settings → Actions → General → Workflow permissions** and enable **Read and write permissions**.
4. Push to the `main` branch and inspect **Actions**. Capture a screenshot showing a successful workflow run and its jobs.
5. Open [Streamlit Community Cloud](https://share.streamlit.io/), connect the GitHub repository, select `main`, and set the app file path to `app/streamlit_app.py`.
6. Confirm `models/best_model.joblib` is committed. Once deployed, test multiple customer profiles and capture a screenshot.
7. Replace the placeholders below with the actual public URLs. Do not submit placeholder links.

### Business Context
This step contributes to a reproducible customer purchase-propensity solution for Visit with Us.

### Objective
Complete the current implementation step in the end-to-end MLOps workflow.

### Purpose of this code
Execute the operation described in the section heading and produce outputs required by subsequent stages.

In [ ]:
GITHUB_REPOSITORY_URL = "PASTE_YOUR_PUBLIC_GITHUB_REPOSITORY_URL_HERE"
STREAMLIT_APP_URL = "PASTE_YOUR_PUBLIC_STREAMLIT_APP_URL_HERE"
print("GitHub repository:", GITHUB_REPOSITORY_URL)
print("Streamlit app:", STREAMLIT_APP_URL)

### Observation and Business Insight
After executing this cell, review the displayed output and record the actual result. Explain what it means for customer targeting, model reliability or pipeline readiness. Do not insert assumed values. Where the cell prints metrics or creates plots, refer to those executed outputs and identify any follow-up action.

## 13. Final business insights and recommendations

Complete this section after reviewing the actual evaluation table and threshold table generated above.

### Recommended business actions
1. **Use propensity for prioritization, not automatic exclusion.** Rank customers by predicted probability and let marketing capacity determine the outreach cutoff.
2. **Align threshold with campaign economics.** If missing a potential buyer is more costly than contacting a non-buyer, choose a lower threshold; if sales capacity is constrained, prioritize precision and a smaller high-propensity segment.
3. **Run a controlled pilot.** Randomly assign eligible customers to treatment and control groups to measure incremental Wellness-package conversion. Predictive propensity is not proof that a campaign causes purchases.
4. **Collect Wellness-specific labels.** Record actual views, inquiries, bookings and purchases for the Wellness Tourism Package. Retrain against that outcome once sufficient representative data is available.
5. **Monitor drift and performance.** Track input distributions, positive-class rate, recall, precision, PR-AUC and calibration after each campaign. Trigger retraining only with validated new data and an approved evaluation gate.
6. **Respect customer consent and privacy.** Use customer information only for permitted marketing purposes, apply access controls, and provide appropriate opt-out mechanisms.

### Final submission checklist
- [ ] All notebook cells executed from top to bottom without errors.
- [ ] Actual EDA, model comparison, classification report and confusion matrix visible.
- [ ] Repository contains dataset, scripts, model, app, dependencies and workflow.
- [ ] GitHub Actions workflow run succeeded; screenshot captured.
- [ ] Streamlit app is public and tested; screenshot captured.
- [ ] Real GitHub and Streamlit URLs inserted above.
- [ ] Notebook exported from Colab as HTML after execution.

## 14. Executive summary – evidence-led conclusions

### Business Context
Visit with Us wants to replace inconsistent manual customer selection with a scalable, repeatable approach to prioritizing customers for a new Wellness Tourism Package.

### Objective
Translate the executed analysis and model evaluation into decisions that marketing leaders can act on.

### Purpose
This section consolidates evidence from the notebook. Complete the bracketed prompts only after the notebook has run successfully.

### Key findings from the executed analysis
- **Data quality:** State the executed row/column count, missing-value count, duplicate count and any removed fields.
- **Customer profile:** Describe the most relevant customer patterns visible in the executed EDA. Avoid claiming a relationship unless the displayed analysis supports it.
- **Purchase behaviour:** Report the actual buyer proportion and explain the class imbalance.
- **Model selection:** Name the model selected by the stated metric and quote its actual holdout PR-AUC, recall, precision and F1 from the evaluation table.
- **Threshold decision:** Recommend a threshold only after considering the executed threshold table and campaign capacity.

### Business insights and recommendations
1. **Prioritize outreach using ranked propensity.** Use predicted probability to rank eligible customers, then select a campaign cohort based on available sales capacity. Do not treat a probability as a guaranteed purchase.
2. **Balance missed opportunities and campaign cost.** If the cost of missing a potential buyer is high, consider a lower threshold and accept more outreach. If agent capacity is constrained, use a higher threshold and monitor the reduction in recall.
3. **Validate incremental impact.** Run a randomized pilot with treatment and control groups. Compare Wellness-package conversion, revenue per contacted customer and cost per incremental conversion. A propensity model predicts likelihood; it does not establish that outreach causes purchase.
4. **Create Wellness-specific outcome labels.** The supplied `ProdTaken` column represents historical package purchase behaviour and does not explicitly identify Wellness Tourism Package purchases. Capture actual Wellness-package inquiry, booking and purchase outcomes, then retrain once sufficient representative labels exist.
5. **Respect the pre-contact decision point.** Keep pitch-stage variables out of the primary pre-contact model. If the business later wants a post-pitch conversion model, build and validate it as a separate use case with an explicitly different prediction time.
6. **Operationalize model monitoring.** Track data drift, buyer prevalence, precision, recall, PR-AUC and calibration over time. Re-evaluate on fresh labeled data before promoting a replacement model.
7. **Maintain responsible customer engagement.** Use customer data only for permitted purposes, restrict access, protect exported data and provide appropriate marketing opt-out mechanisms.

### Limitations
- The target is a proxy for historical package purchase, not a Wellness-specific outcome.
- The supplied data may not represent future customers or campaign conditions.
- Model scores depend on the historical sample and should be monitored after deployment.
- The app is a technical demonstration; business rollout requires consent, fairness, privacy and campaign-impact checks.

### Final submission evidence
Insert the actual public GitHub repository URL, successful GitHub Actions run screenshot, public Streamlit Community Cloud URL and deployed-app screenshot. Do not use placeholder URLs or screenshots of an unexecuted workflow.